

# Diagnóstico de componentes: `DedupePyFunc` concentra 6 capas (11 componentes)

El script no separa capas por archivo ni por clase: todo vive en un solo módulo y el 70% del comportamiento está dentro de un único método, `_dedupe_one`. Este es el mapa componente por componente.

| # | Capa / Componente | Funciones y métodos que la ocupan | Depende de |
| --- | --- | --- | --- |
| 1 | **Ingesta / parsing** | `DedupePyFunc.predict`, `normalize_payload` | `json`, `pandas`; contrato de entrada = DataFrame con columna `payload` |
| 2 | **Extracción** (evento → cliente canónico) | `extract_dedupe_client` (orquestador), `_get_group`, `_changes_to_dict`, `_pick_address`, `_pick_phone`, `_pick_email`, `_infer_pnp`, `_infer_customer_type`, `_infer_es_fideicomiso`, `_build_full_name`, `_build_address_str` | Solo `typing`. No usa la capa de normalización: entrega texto crudo |
| 3 | **Normalización / limpieza** | Núcleo: `_safe_str`, `strip_accents`, `normalize_spaces`. Texto: `clean_name_v2`, `clean_address_v2`. Identificadores: `clean_tax_id`, `clean_curp`, `clean_lei_code`, `clean_foreign_fiscal_id`, `clean_foreign_fiscal_country`, `clean_foreign_fiscal_key`, `clean_efirma`. Contacto/otros: `clean_email`, `norm_phone`, `norm_postal`, `norm_date_str`, `clean_cp_fiscal`, `clean_razon_social`, `clean_suf_razon_social`, `clean_regimen_fiscal`, `clean_es_fideicomiso`, `clean_customer_type`, `clean_customer_type_input`, `map_customer_type_label` | `re`, `unicodedata`, `numpy`, `pandas`. Constantes: `_ws`, `_punct`, `_taxid_keep`, `ADDR_REPL`, `STOPWORDS_ADDR`, `STOPWORDS_RAZON_SOCIAL`, `CUSTOMER_TYPE_MAP` |
| 4 | **Validación de dominio** | `is_valid_rfc_basic` (`_rfc_pf`, `_rfc_pm`), `is_valid_curp_basic`, `is_fideicomiso_name`, `is_foreign_customer_type` | Capa 3  Verificar que los identificadores ya normalizados sigan las reglas fiscales y de negocio |
| 5 | **Carga de estado / artefactos** | `DedupePyFunc.load_context` (incluye `_map_cls` anidada) | Artefactos: `customers_base` (parquet), `customer_ids`, `emb_full_name`, `emb_direccion`, `beto_dir`. Usa capa 3 + `CLASSIFICATION_MAP`. Produce `base_df`, `base_pnp`, `base_tax`, `base_name`, `base_curp`, `base_lei`, `base_foreign_fiscal_key`, `E_NAME`, `E_ADDR`, `tokenizer`, `model`, `self.p` |
| 6 | **Encoding** | `encode_one`, `mean_pooling` | `torch`, `transformers` (`AutoTokenizer`, `BertModel`), estado de capa 5, params `max_len_name` / `max_len_addr` |
| 7 | **Retrieval / blocking** | `search_topk`, `compute_addr`, más dos rutas inline en `_dedupe_one`: el bloque de RFC parcial (`np.fromiter` + reordenamiento) y la ruta fideicomiso (`np.where(base_pnp == group_key)`) | `numpy`, capas 5 y 6, params `faiss_k` y `chunk_size` |
| 8 | **Scoring / features** | Loop `for i, rid in enumerate(rows)` dentro de `_dedupe_one` (base_score + `boost_total` + `feature_flags`), `fuzzy_name_score`, `fuzzy_addr_score` | `rapidfuzz`, capas 3, 4 y 7, pesos y boosts de `self.p` |
| 9 | **Decisión / política** | Todo inline en `_dedupe_one`: hard rules de `customer_number` y `tax_id`, umbrales `dup_threshold` / `review_threshold`, override `force_duplicate_foreign_top`, downgrade por `strong_support_count`, conversión `actualizacion → coincidencia` | Capa 8, `pandas` para `pd.isna` |
| 10 | **Explicabilidad / respuesta** | `build_business_explanation`, `build_candidates`, y los dicts `explanation` / `candidate_explanations` armados inline | Capa 8 (flags y components) y `base_df` de capa 5 |
| 11 | **Serving wrapper** | Clase `DedupePyFunc(mlflow.pyfunc.PythonModel)` = `load_context` + `predict` | `mlflow.pyfunc`, `time` |

### Flujo de dependencias

`predict` → `normalize_payload` → `extract_dedupe_client` (capa 2) → `_dedupe_one`, que dentro suyo encadena normalización (3) → validación (4) → encoding (6) → retrieval (7) → scoring (8) → decisión (9) → explicación (10), leyendo siempre el estado que dejó `load_context` (5).

### Hallazgos de arquitectura

`_dedupe_one` es un god-method: absorbe él solo las capas 7, 8, 9 y 10, incluyendo lógica de retrieval que debería estar en funciones (el bloque de RFC parcial y la ruta de fideicomiso están escritos a mano dentro del método). Es el punto que hay que partir primero.

Hay tres definiciones de `_safe_str` y `strip_accents`, y dos de `clean_curp`; solo la última de cada una queda activa, y las versiones difieren en comportamiento (por ejemplo, la primera `_safe_str` no reconoce `<NA>` y la última sí). Las constantes `_keep_addr` y `_email_split`, y las funciones `clean_foreign_fiscal_id`, `clean_foreign_fiscal_country`, `clean_lei_code`, `clean_customer_type` y `map_customer_type_label` están definidas pero nunca se invocan; la limpieza de esos campos se hace en otro lado o se asume ya hecha en el parquet.

La capa de extracción (2) entrega campos crudos y la capa 5 asume que el parquet ya trae columnas `*_clean` precalculadas (`email_home_clean`, `phone_cell_clean`, `cp_fiscal_clean`, `efirma_clean`, `regimen_fiscal_clean`, entre otras) que este script nunca genera. Es decir, la normalización está partida entre el runtime y un proceso de construcción de base que vive fuera del archivo: acoplamiento implícito y sin contrato.

Los ~30 hiperparámetros (`self.p`) están hardcodeados en `load_context` en vez de venir de una configuración externa, y `predict` permite sobreescribirlos desde el payload sin validación. No hay esquemas de entrada/salida ni logging en ninguna capa.

---

# Integración con el framework

Al cruzar el diagnóstico interno con el entorno donde vivirá el modelo, revisé `models/predictive_classical_ml/antifraude_rf/` (los tres archivos), el contrato que valida el framework, `runner.run_zenml`, el DAG y `shared_steps/tracking.py`. La conclusión corta: **la analogía sí cierra, y el punto de BERT ya está resuelto por diseño en el framework** — `tracking_run` detecta si el objeto que devuelve `train_model` ya es un `mlflow.pyfunc.PythonModel` y lo registra tal cual, sin doble envoltura. Pero hay cuatro puntos donde el framework, como está hoy, bloquea el modelo experto.

## Cómo llama el framework al plugin

`run_zenml(plugin_ref)` resuelve el tipo de pipeline por la carpeta (`models/<tipo>/<plugin>/` ↔ `src/pipelines/<tipo>/`), valida el contrato por AST, carga la configuración como `plugin.CONFIG`, adjunta `schemas.py` como `plugin.SCHEMAS` y lanza el DAG. El DAG solo encadena tres funciones del plugin — `load_split`, `train_model`, `validate_model` — y una cuarta pieza compartida, `tracking_and_promote_step`, que registra, compara contra baseline y promueve.

El validador comprueba **aridad posicional, no nombres**. Eso te da libertad para renombrar los parámetros al vocabulario del modelo experto.

## Mapeo del ciclo de vida

| Pieza del contrato | En `antifraude_rf` | En el modelo experto de deduplicación |
| --- | --- | --- |
| `experiment_name` | `antifraude_rf-v1` | `dedupe_clientes-v1` |
| `data.splits.train` | `gold_train.csv` | **Set de calibración**: los casos históricos con dictamen experto sobre los que se fijaron en su momento los umbrales, pesos y boosts. Se declara por linaje, para dejar constancia de qué datos justifican los parámetros. El paso de entrenamiento lo recibe y no lo usa |
| `data.splits.calibration` | `null` | Se deja en nulo. La calibración ya ocurrió, fue manual y su resultado son los parámetros del config |
| `data.splits.validation` | `gold_validation.csv` | **Set de validation**: los casos con los que negocio verifica que las reglas se comportan. Es el único conjunto que realmente se ejecuta |
| `target_column` | `target` | `decision_experto`: la columna con el dictamen. Solo se usa en validación, para comparar contra lo que decidió el motor |
| `model_params.best_params` | Hiperparámetros del RandomForest | Los parámetros de calibración completos: `dup_threshold`, `review_threshold`, los cuatro pesos `w_*`, los quince `boost_*`, `faiss_k`, `chunk_size`, `max_len_name`, `max_len_addr`. Son el equivalente a los coeficientes: lo que se versiona y lo que viaja |
| `model_params.optimal_threshold` | Umbral ROC | `dup_threshold`, replicado con ese nombre porque el framework lo lee explícitamente |
| *(nuevo)* `model_params.artifacts` | — | El diccionario de artefactos, declarado en el `config.yaml`. Cinco claves fijas, cuatro archivos y una carpeta:<br>`customers_base`, `customer_ids`, `emb_full_name`, `emb_direccion`, `beto_dir`.<br>Es **la misma estructura** que el modelo recibirá después desde el contexto, así que sirve para los dos caminos: cuando corre el ciclo lee de estas rutas, y cuando lo carga el registro lee de las rutas reescritas |
|
| `load_split` | Lee el CSV y valida su estructura | Igual: lee el conjunto de casos etiquetados y valida su estructura. Se llama dos veces, una por conjunto. La base de clientes no entra por aquí, es artefacto del modelo |
| `train_model` | `fit` del Pipeline sklearn | **No entrena y no ajusta nada.** Toma los parámetros que el framework leyó del config, instancia el motor con ellos y con las referencias a los artefactos, y lo devuelve. Ese objeto es lo que se validará y lo que se registrará. |
| `validate_model` | ROC AUC + classification report + figura | Recibe el motor que devolvió el paso anterior, carga lo que necesita para ejecutar, corre el golden set y compara la decisión contra el dictamen experto. Devuelve precisión y recall de duplicado, tasa de envío a revisión, cobertura de reglas duras y tasa de override extranjero, más la matriz de confusión como artefacto |
| `schemas.py` | `PredictRequest` tabular / `PredictResponse` con dos probabilidades | `PredictRequest` con el payload del evento; `PredictResponse` con `decision`, `score`, `reason`, `business_explanation` y `candidates` |
| `mlflow.compare_keys` | `roc_auc`, `f1`, `accuracy` | `precision_duplicado`, `recall_duplicado`, `tasa_revision` |



## El punto crítico: dónde vive BERT

Tu intuición es correcta y además es la regla explícita del framework. `src/serving/runtime.py` lo dice así: *"si al dato hay que hacerle algo, eso es parte del MODELO y viaja dentro de él desde `build_model`"*. BETO no es un modelo del ciclo de vida, es una **dependencia interna del motor de reglas**, en el mismo sentido en que el `ColumnTransformer` vive dentro del `Pipeline` de `antifraude_rf` y la decodificación base64 vive dentro de `OcrModel.predict`. Así que sí: BERT y la PyFunc conviven en el mismo objeto, y `train_model` devuelve ese objeto.

El mecanismo que lo hace viable son dos líneas de `shared_steps/tracking.py`:

```161:180:src/shared_steps/tracking.py
def _as_pyfunc(model: Any) -> mlflow.pyfunc.PythonModel:
    if isinstance(model, mlflow.pyfunc.PythonModel):
        return model
    return PluginModel(model)
```

Como `DedupePyFunc` ya hereda de `mlflow.pyfunc.PythonModel`, pasa sin envolverse (importante: el envoltorio genérico `PluginModel` llama a `predict_proba`, que tu motor no tiene). Y justo antes, `_pickle_by_value` registra el módulo de `type(model).__module__` — es decir, el `model.py` de tu propio plugin — para que cloudpickle serialice la clase **por valor** dentro del artefacto. Eso es lo que permite que un monolito como el tuyo se registre como un solo modelo servible.

## Los cuatro bloqueos reales

**1. `log_model` se llama sin `artifacts`.** Es el bloqueo más duro. `tracking_run` invoca `mlflow.pyfunc.log_model(name="model", python_model=..., signature=...)` y nada más, así que `context.artifacts` llegará vacío y tu `load_context` revienta en la primera línea. Cargar todo en `__init__` para que cloudpickle lo serialice no es opción con BETO más la base más dos matrices de embeddings. Hace falta un passthrough de artefactos en `tracking_run` (parámetro nuevo que se propague a `log_model`), declarado desde `model_params.artifacts` del config.

**2. Los steps de ZenML están tipados a sklearn.** `train_step` devuelve `Annotated[Pipeline, "champion_pipeline"]` y `validate_step` recibe `trained_pipeline: Pipeline` y `df_validation: pd.DataFrame`. ZenML valida esas anotaciones, y un `DedupePyFunc` no las cumple. La salida limpia es crear el tipo de pipeline propio (`models/expert_rules_nlp/` + `src/pipelines/expert_rules_nlp/` con su `pipeline.py`, sus `steps/` en `Any` y su validador), en lugar de forzar el modelo dentro de `predictive_classical_ml`.

**3. El contrato de serving no encaja con tu `predict`.** `deduce_input_format` decide por número de campos del `PredictRequest`: un solo campo entrega `list[str]` en crudo, dos o más entregan un DataFrame. Tu `predict` hace `model_input.iloc[0]["payload"]`, o sea asume DataFrame y además procesa **solo la primera fila**. Y `to_responses` exige una predicción por registro de entrada, mientras que hoy devuelves una única fila con todos los resultados dentro. Hay que reescribir `predict` para que acepte ambos envoltorios y devuelva una salida por registro, en el orden de los campos del `PredictResponse`.

**4. Las dependencias pesadas no se declaran.** `log_model` no recibe `pip_requirements` ni `conda_env`, así que `torch`, `transformers` y `rapidfuzz` no quedan en el `MLmodel`. El servicio fusionado de BentoML se despliega asumiendo solo `bentoml`, `mlflow`, `pandas` y `pydantic`. Para este plugin eso hay que resolverlo explícitamente en la imagen de serving.

Un quinto punto que no es bloqueo pero sí decisión de arquitectura: si la base de clientes y las dos matrices de embeddings viajan como artefactos del modelo, cada corrida pesa varios GB y el reentrenamiento del vocabulario obliga a re-registrar todo. La alternativa es versionar como artefacto solo BETO y los parámetros, y referenciar base y embeddings por URI con su huella de contenido.

---

# Focos rojos: lo que no cuadra del script actual

**Hiperparámetros embebidos y sobreescribibles sin control.** Los ~30 parámetros de calibración (umbrales, los cuatro pesos, los quince boosts, longitudes de tokenización) están escritos dentro de `load_context`, así que cambiar un umbral es cambiar código. Y hay algo peor: `predict` hace `p_req.update(params_override or {})` con lo que venga en el payload, sin validar nada. Hoy cualquiera que llame al modelo puede mover `dup_threshold` desde el request. Eso hay que sacarlo a configuración y cerrar el override.

**Falso batching en `predict`.** Recibes un DataFrame y lees `model_input.iloc[0]["payload"]`. Dentro de ese payload sí iteras todos los `events`, pero las filas 2 a N del batch se descartan en silencio. Y el resultado sale como una sola fila con una lista de resultados dentro, cuando el contrato de serving espera una respuesta por registro de entrada. Son dos fallas distintas y ambas hay que arreglar.

**Desincronización entre la limpieza y los embeddings.** Los vectores de `emb_full_name.npy` y `emb_direccion.npy` se generaron codificando textos limpiados con una versión concreta de `clean_name_v2` y `clean_address_v2`. Si tocas `ADDR_REPL` o las stopwords, la consulta se normaliza con reglas nuevas y la base sigue codificada con las viejas: comparas vectores de dos preprocesamientos distintos y nada lo detecta. En segundo plano está el mismo problema con las columnas `*_clean` del parquet, que este script no genera y que alimentan las señales exactas.

**No hay FAISS.** `search_topk` recorre la matriz de embeddings por bloques con producto punto de numpy. El parámetro `faiss_k` es un residuo del nombre y solo significa cuántos candidatos traer. Conviene renombrarlo antes de que alguien asuma que existe un índice vectorial.

**El score no es una probabilidad.** Es una suma ponderada de similitudes más boosts, truncada con `min(..., 1.0)`. No está calibrado, así que ni las métricas de gobernanza ni la explicación pueden tratarlo como probabilidad.

**Inconsistencia entre lectura tolerante y dura de la base.** En el scoring, columnas como `email_home_clean` y `phone_home_clean` se leen con `.get(..., "")`, así que su ausencia solo apaga el boost. Pero `build_candidates` las lee con acceso directo. Si el parquet no las trae, el modelo puntúa sin quejarse y luego truena al construir la respuesta, en el momento más confuso posible.

---

# Flujo del motor de inferencia

| Etapa | Componente(s) | Rol en la arquitectura |
| --- | --- | --- |
| **Carga, validación de datos y entrenamiento** | — | No existen en este script: es exclusivamente el motor de inferencia. |
| **Inicialización** | Carga de artefactos | Al levantar el servicio se cargan la base de clientes, las dos matrices de embeddings, el tokenizer y los pesos de BERT, y se fijan los parámetros de calibración. No hay scalers ni vocabularios: no existen en este modelo. |
| **Preprocesamiento** | Extracción del evento, normalización, encoding | Al llegar la petición se aplana el evento a un cliente canónico, se normalizan sus campos y se codifican nombre y dirección con BERT. La limpieza tiene que usar exactamente las mismas reglas con las que se generaron los embeddings de la base. |
| **Inferencia, fase 1** | Retrieval / blocking | Filtra la base por tipo de persona y recorre la matriz de nombres por bloques para quedarse con los candidatos más parecidos. Además inyecta a la fuerza los que comparten prefijo de RFC, que podrían no haber entrado por similitud. |
| **Inferencia, fase 2** | Scoring | Por cada candidato combina similitud semántica de nombre y dirección, similitud fuzzy y boosts por coincidencias exactas de RFC, CURP, correo, teléfono, fechas e identificadores fiscales. La salida es un score de reglas, no una probabilidad. |
| **Decisión política** | Umbrales y reglas duras | Los umbrales de duplicado y revisión clasifican el score. Ojo con el orden: las reglas duras de `customer_number` y RFC exacto van **antes** del scoring y cortocircuitan todo el flujo. Después vienen el override por identificación fiscal extranjera y la degradación a revisión cuando faltan señales fuertes de apoyo. |
| **Explicabilidad** | Explicación de la respuesta | Texto de plantilla armado a partir de las banderas de coincidencia. Los desgloses por candidato se construyen durante el scoring, no al final. Nada de SHAP ni de atribución de features. |
| **Serving** | Service wrapper | Artefacto `pyfunc` de MLflow más el endpoint que genera el framework a partir del esquema declarado. No es un microservicio en FastAPI ni Flask escrito a mano. |

## Aclaraciones sobre los componentes

**Retrieval / blocking.** Sirve para no aplicar el modelo pesado a toda la base. Aquí el ahorro es doble: BERT solo codifica la **consulta**, porque los vectores de los clientes existentes están precalculados en los `.npy`, y el filtro por tipo de persona descarta de entrada la mitad de la base. De los candidatos que sobreviven, se conserva un top de unos cientos y solo esos pasan al scoring detallado. Este componente sí es necesario en tu caso: sin él tendrías que comparar campo por campo contra millones de clientes por cada evento.

**Decisión política.** Es tu capa de negocio pura: transforma el score en una acción del sistema (`duplicado`, `revisar`, `no_duplicado`, `coincidencia`). Con un matiz importante en tu diseño: no toda la política vive al final. Las reglas duras de `customer_number` y de RFC exacto se evalúan al principio y devuelven de inmediato, sin pasar por embeddings ni scoring.

**Explicación de la respuesta.** Es la capa de explicabilidad, y en tu modelo es determinista: cada boost que se dispara enciende una bandera, y al final las banderas se traducen a una frase en lenguaje de negocio. No hay atribución estadística, y eso es una ventaja para un sistema experto auditable.

**Service wrapper.** Es el caparazón que carga los artefactos en memoria al arrancar y enruta cada petición por preprocesamiento, retrieval, scoring, decisión y explicación. En este framework son dos piezas y ninguna la escribe el científico de datos: el artefacto `pyfunc` de MLflow y el endpoint que se genera a partir del esquema declarado

## Tabla 2 — Ciclo de Vida MLOps para el Sistema Experto

| Etapa del Ciclo de Vida | Componente(s) Involucrados | Rol / Implementación (Analogía ML) |
| --- | --- | --- |
| **1. Carga de datos** | Conjunto de calibración y conjunto de validación | Se cargan los dos conjuntos en memoria y se valida su estructura, con una llamada al mismo cargador por cada uno:<br>1. El conjunto donde se calibraron las reglas.<br>2. El conjunto que dio negocio para validar las reglas. |
| **2. Entrenamiento (calibración e instanciación)** | 1. Configuración<br>2. Clase nativa<br>3. Normalización<br>4. Retrieval<br>5. Scoring<br>6. Decisión política<br>7. Explicabilidad<br>8. Modelo BERT (bloque de encoding) | **No hay un `fit()` matemático.** La etapa recibe los hiperparámetros ya calibrados por los expertos y con ellos instancia la clase nativa, que ensambla dentro los componentes 3 al 8.<br><br>Lo que hace el paso, en orden:<br>1. Recibe de la configuración los hiperparámetros **y el diccionario de artefactos** con las rutas en disco.<br>2. Instancia la clase nativa pasándole ambos.<br>3. La clase guarda los parámetros y las rutas. No abre ningún archivo todavía: solo texto, pesa nada.<br>4. Devuelve esa instancia, que es la que se validará y la que se registrará.<br><br>La clase que aquí se instancia no es solo el motor de decisión: es el **modelo servible completo**, porque implementa el contrato estándar de MLflow con su método de inicialización y su método de predicción.<br><br>**De momento es un monolítico** donde conviven dos modelos en la misma ejecución: **BERT**, al que el motor llama internamente para codificar nombre y dirección, y el **motor de inferencia**, que decide.<br><br>Opcionalmente, aquí se puede barrer el umbral de duplicado sobre el conjunto de calibración: eso convierte la calibración experta en un paso reproducible y versionado, y le da contenido real al "entrenamiento". |
| **3. Validación del modelo** | Motor de inferencia + conjunto de validación | Se ejecuta sobre el conjunto de validación el mismo objeto que devolvió la etapa 2. Aquí el motor **abre por primera vez los archivos**, usando las rutas de disco que trae de la configuración: la base, las dos matrices y los pesos de BERT. Corre el golden set, compara su decisión contra el dictamen experto y devuelve precisión y recall de duplicado, tasa de envío a revisión y la matriz de confusión. |
| **4. Registro y empaquetado** | Clase nativa del plugin, que implementa el contrato pyfunc | El wrapper se define ** en el model.py en el plugin,la clase expone el método de predicción con la firma estándar y el de inicialización que recibe los artefactos, así que el framework la registra tal cual y no le aplica su wrapper  genérico.<br>Al registrar se pasa **el mismo diccionario de artefactos**, y MLflow copia esos cinco elementos dentro de la carpeta del modelo. Desde ese momento el modelo es autocontenido.<br>Salvedad: hoy el framework registra sin adjuntar artefactos, así que este paso está bloqueado hasta que se le agregue. |
| **5. Gobernanza** | Métricas de la etapa 3 | El framework compara las métricas contra el baseline registrado y, si las iguala o supera, promueve el modelo a champion. En la primera corrida no hay baseline: esa ejecución es la que lo establece. |
| **6. Serving** | Endpoint expuesto | El framework publica la ruta de inferencia a partir del esquema declarado y sirve la última versión promovida. Al cargar, el modelo recibe **el mismo diccionario, con las mismas claves pero las rutas reescritas** hacia la copia local, y con eso abre la base, las matrices y BERT. Ocurre una sola vez por réplica, en la primera petición. |

 |

Sí, el cuerpo del motor se pasa tal cual: `_dedupe_one` completo, las funciones de limpieza, validación, encoding, retrieval, scoring, decisión y explicación. Nada de eso se toca.

Los cambios son en los bordes de la clase, y son cuatro:

**Constructor y método de carga.** Hoy `load_context` hace las dos cosas a la vez: fija los parámetros y abre los archivos. Hay que partirlo en un constructor que reciba parámetros y rutas, y un método de carga que acepte rutas opcionales; `load_context` queda como una línea que delega pasándole las del contexto. Sin esto no puedes validar antes de registrar.

**Los parámetros salen del constructor, no del código.** El bloque de treinta valores desaparece de `load_context` y llega como argumento.

**`predict` procesa todas las filas y devuelve una respuesta por registro**, en lugar de leer solo la primera y devolver una fila con todo adentro. Y se quita el override de parámetros desde el payload.

**Excluir el estado pesado de la serialización**, para que el objeto cruce entre pasos y se registre pesando kilobytes en vez de gigabytes. Son unas pocas líneas declarando qué atributos no se serializan.

Más las tres funciones del contrato, que van fuera de la clase y son cortas: el cargador de conjuntos, el que instancia y devuelve el motor, y el que corre el golden set y calcula métricas.

Lo demás —las definiciones duplicadas, el nombre `faiss_k`, los barridos por petición sobre toda la base— son mejoras, no bloqueos. Se pueden dejar para después sin que nada deje de funcionar.

Es una regla que le dice a Python **qué NO guardar cuando serializa el objeto**.

El problema: después de la validación, tu instancia tiene pegados como atributos la base, las dos matrices, el tokenizer y los pesos de BERT. Cuando el orquestador la pasa al siguiente paso, o cuando MLflow la registra, Python guarda **todos** los atributos sin preguntar. Ahí es donde se te van los gigabytes al disco.

La solución son estas líneas dentro de la clase:

```python
_PESADOS = ("base_df", "E_NAME", "E_ADDR", "tokenizer", "model",
            "base_pnp", "base_tax", "base_curp", "base_lei")

def __getstate__(self):
    estado = self.__dict__.copy()
    for atributo in _PESADOS:
        estado.pop(atributo, None)
    return estado
```

`__getstate__` es un método que Python llama solo, automáticamente, cada vez que serializa el objeto. Tú nunca lo invocas. Lo que devuelva es lo que se guarda. Aquí devuelve todo **menos** los pesados, así que lo que queda son los parámetros y las rutas: unos kilobytes.

Cuando el objeto se restaura del otro lado, esos atributos simplemente no están, y el método de carga los reconstruye leyendo los archivos. Que es justo lo que ya hace tu código.

Es como guardar una partida de videojuego: guardas en qué nivel vas y tu inventario, no las texturas ni el motor gráfico. Eso se vuelve a cargar del disco cuando retomas.



# Integración de DedupePyFunc

# Cambios de la clase

## Cambios generales

La clase conserva el nombre DedupePyFunc. Los métodos nuevos deben utilizar nombres en inglés que describan su acción: load_artifacts, extract_client, normalize_payload, process_record, build_candidates y build_business_explanation.

Los parámetros y variables deben indicar qué contienen. Sustituir p por calibration_config, a por artifact_paths, c por customer_record, b por candidate_row, base_df por df_customers, E_NAME por name_embeddings, E_ADDR por address_embeddings y model por bert_model.

Los docstrings pueden permanecer en español y deben incluir descripción, parámetros, retorno y excepciones.

## 1. Agregar un constructor para la calibración

El diccionario de calibración no debe crearse dentro de load_context. Debe recibirse cuando se crea la instancia:

```python
class DedupePyFunc(mlflow.pyfunc.PythonModel):
    def __init__(
        self,
        calibration_config: dict[str, Any],
    ) -> None:
        self.calibration_config = {
            **DEFAULT_CALIBRATION,
            **calibration_config,
        }

```

El constructor solo guarda la calibración. La carga de archivos se mantiene en un método separado para que el momento de construcción y el momento de carga sean explícitos.

## 2. Sustituir load_context por load_artifacts

Todo el cuerpo actual de load_context que abre archivos debe moverse a:

```python
def load_artifacts(
    self,
    artifact_paths: dict[str, str],
) -> None:
    self.df_customers = pd.read_parquet(
        artifact_paths["customers_base"]
    )
    customer_ids = np.load(
        artifact_paths["customer_ids"],
        allow_pickle=True,
    ).astype(str)
    self.name_embeddings = np.load(
        artifact_paths["emb_full_name"],
        mmap_mode="r",
    )
    self.address_embeddings = np.load(
        artifact_paths["emb_direccion"],
        mmap_mode="r",
    )
    self.tokenizer = AutoTokenizer.from_pretrained(
        artifact_paths["beto_dir"],
        local_files_only=True,
        use_fast=True,
    )
    self.bert_model = BertModel.from_pretrained(
        artifact_paths["beto_dir"],
        local_files_only=True,
        add_pooling_layer=False,
    ).to(self.device)

```

load_context debe eliminarse. load_artifacts se ejecuta una sola vez.

## 3. Extraer toda la calibración

Los 25 valores actualmente asignados en self.p deben moverse a config.yaml. También deben parametrizarse los valores escritos directamente dentro del scoring: Umbrales 0.80, 0.78, 0.75, 0.85 y 0.90, e Incrementos 0.08, 0.06 y 0.10.

Ejemplo:

```python
```yaml
  model_params:
    top_k_candidates: 200,
    top_k_output: 5,
    retrieval_chunk_size: 20_000,
    duplicate_threshold: 0.91,
    review_threshold: 0.86,
    personal_name_embedding_min: 0.80,
    personal_name_fuzzy_min: 0.78,
    personal_identity_boost: 0.08,
  artifacts:
    customers_base: path/to/customers_base.parquet
    customer_ids: path/to/customer_ids.npy
    emb_full_name: path/to/emb_full_name.npy
    emb_direccion: path/to/emb_direccion.npy
    beto_dir: path/to/beto

```

Los métodos deben leer self.calibration_config. El parámetro p debe eliminarse de _dedupe_one.

## 4. Corregir la predicción por lote

La firma debe ser:

```python
def predict(
    self,
    context: Any,
    model_input: Any,
    params: dict[str, Any] | None = None,
) -> list[tuple]:

```

El acceso actual solo procesa la primera fila. Debe recorrerse todo model_input y devolverse una salida por fila:

```python
def predict(self, context, model_input, params=None) -> list[tuple]:
    records = (
        model_input.to_dict("records")
        if isinstance(model_input, pd.DataFrame)
        else list(model_input)
    )
    return [self.process_request(record) for record in records]

```

predict no debe llamar load_artifacts.

## 5. Eliminar parámetros enviados dentro del payload

normalize_payload debe devolver únicamente los eventos. Deben eliminarse params_override y p_req.update(params_override or {}). Los umbrales de negocio provienen de la configuración con la que se construyó la instancia. No deben modificarse desde una petición.

## 6. Definir una entrada por fila

Cada fila de model_input debe representar un evento. El formato alternativo con eventos anidados debe eliminarse o rechazarse. Esto garantiza una relación directa N filas de entrada a N resultados. Si después se necesita admitir varios eventos dentro de una fila, deberá definirse otro contrato de salida que devuelva una lista por fila.

## 7. Validar columnas del parquet

Antes de construir columnas derivadas, debe validarse la presencia de las columnas obligatorias. El patrón df_customers.get("curp_clean", "").fillna("") falla cuando la columna no existe. Para columnas opcionales:

```python
def optional_string_column(
    df_customers: pd.DataFrame,
    column_name: str,
) -> pd.Series:
    if column_name in df_customers:
        return df_customers[column_name].fillna("").astype(str)
    return pd.Series("", index=df_customers.index, dtype="string")

```

Debe aplicarse el mismo criterio durante el scoring y al construir candidatos.

## 8. Renombrar componentes internos

_dedupe_one debe renombrarse como process_record. Sus argumentos deben ser customer_record y debe utilizar self.calibration_config. También deben renombrarse los atributos mencionados en Cambios generales para evitar nombres de una letra y abreviaturas sin contexto.

## Mejoras sugeridas de la clase

Consolidar funciones duplicadas conservando una sola definición de strip_accents, _safe_str y clean_curp, borrando funciones y expresiones regulares no utilizadas. Renombrar faiss_k cambiándolo a top_k_candidates ya que el código no utiliza FAISS. Precalcular búsquedas creando una vez el índice del número de cliente normalizado y los RFC agrupados por prefijo. Filtrar fideicomisos para reducir candidatos antes de puntuar todo el grupo de persona moral. Unificar RFC para que clean_tax_id e is_valid_rfc_basic apliquen la misma política sobre Ñ y &. Eliminar cálculos sin uso eliminando o utilizando explícitamente razon_social_in, suf_razon_social_in y los arreglos derivados que actualmente no participan en decisiones. Aplicar encoding por lote sustituyendo encode_one por una función que reciba una lista de textos. Separar responsabilidades internas dividiendo process_record en recuperación, scoring, decisión y explicación. Implementar logging y pruebas registrando tiempos y conteos sin datos personales, probando lotes, serialización completa, columnas opcionales, RFC con &, fideicomisos y correspondencia de entrada y salida.

# Cambios en las funciones del plugin

## 1. load_split

Debe leer el conjunto indicado, comprobar que exista y devolver un DataFrame:

```python
def load_split(split_uri: str | Path) -> pd.DataFrame:
    split_path = Path(split_uri)
    if not split_path.exists():
        raise FileNotFoundError(split_path)

    if split_path.suffix == ".parquet":
        return pd.read_parquet(split_path)
    if split_path.suffix == ".csv":
        return pd.read_csv(split_path)
    raise ValueError(f"Formato no soportado: {split_path.suffix}")

```

Esta función carga los conjuntos de calibración y validación. No carga la base maestra ni BERT.

## 2. train_model

train_model crea la instancia de DedupePyFunc, separa calibración y rutas, carga todos los recursos y devuelve la instancia completamente preparada:

```python
def train_model(
    df_train: pd.DataFrame,
    target_column: str,
    champion_model_config: dict[str, Any],
) -> DedupePyFunc:
    artifact_paths = champion_model_config["artifacts"]
    calibration_config = {
        name: value
        for name, value in champion_model_config.items()
        if name != "artifacts"
    }

    pyfunc_model = DedupePyFunc(calibration_config)
    pyfunc_model.load_artifacts(artifact_paths)
    return pyfunc_model

```

El objeto devuelto ya contiene el parquet, las matrices, el tokenizer y BERT. Ese mismo objeto se utiliza en los pasos siguientes. df_train no ajusta automáticamente las reglas. Puede utilizarse para validar la estructura del conjunto de calibración.

## 3. validate_model

Debe recibir exactamente la instancia que devolvió train_model. No crea otra instancia ni vuelve a cargar recursos:

```python
def validate_model(
    df_validation: pd.DataFrame,
    target_column: str,
    trained_model: DedupePyFunc,
    optimal_threshold: float,
) -> dict[str, dict[str, Any]]:
    predictions = trained_model.predict(
        context=None,
        model_input=df_validation,
    )
    return {
        "metrics": {
            "agreement_rate": ...,
            "validation_records": float(len(df_validation)),
        },
        "artifacts": {},
    }

```

La función compara las decisiones con target_column y devuelve métricas numéricas. optimal_threshold debe eliminarse si duplica el umbral guardado en calibration_config.

## 4. config.yaml

Debe contener calibración y rutas:

```yaml
model_params:
  duplicate_threshold: 0.91
  review_threshold: 0.86
  top_k_candidates: 200
  artifacts:
    customers_base: path/to/customers_base.parquet
    customer_ids: path/to/customer_ids.npy
    emb_full_name: path/to/emb_full_name.npy
    emb_direccion: path/to/emb_direccion.npy
    beto_dir: path/to/beto

```

Las llaves de artefactos deben coincidir exactamente con las utilizadas por load_artifacts en la clase.



## 5. `schemas.py`

El script recibe eventos anidados (`groups` / `changes`) y devuelve estructuras anidadas (`explanation`, `candidates`, `candidate_explanations`). Mientras el conversor del framework no admita objetos anidados (`src/utils/utils.py` solo mapea `string`, `integer`, `number`, `boolean`, `array`), el contrato de serving debe aplanar lo que el conversor no puede representar.

### Entrada: `PredictRequest`

Un solo campo. El serving entrega una lista de strings (un JSON por petición), no un DataFrame con columnas planas.

```python
from pydantic import BaseModel, Field


class PredictRequest(BaseModel):
    """Evento de deduplicación serializado como JSON (un evento por petición)."""

    payload: str = Field(
        ...,
        min_length=2,
        description="Evento JSON con groups/changes del cliente a evaluar.",
    )

    model_config = {
        "json_schema_extra": {
            "example": {
                "payload": (
                    '{"transaction":"TX-001","client_id":"12345",'
                    '"actualizacion":false,'
                    '"groups":[{"group":"CUSTOMERS","changes":[{'
                    '"field":"rfc","value":"XAXX010101000"'
                    '},{"field":"curp","value":"XAXX010101HDFXXX00"},'
                    '{"field":"personal_type","value":"Persona Física"},'
                    '{"field":"fullName","value":"JUAN PEREZ LOPEZ"}]}]}'
                )
            }
        }
    }
```

**Qué valida el esquema:** que llegue un string no vacío.

**Qué no valida el esquema (lo valida la clase tras `json.loads`):** RFC, CURP, `P`/`N`, `customer_type`, fideicomiso, etc. Esas reglas viven en `extract_client` / `process_record`, no en Pydantic de serving.

### Salida: `PredictResponse`

Campos escalares tipados donde aplica; estructuras anidadas serializadas como JSON en string.

```python
from typing import Literal

from pydantic import BaseModel, Field


class PredictResponse(BaseModel):
    decision: Literal[
        "duplicado",
        "revisar",
        "no_duplicado",
        "coincidencia",
    ] = Field(..., description="Veredicto del motor.")

    score: float = Field(
        ...,
        ge=0.0,
        le=1.0,
        description="Score híbrido del mejor candidato.",
    )

    reason: str = Field(..., min_length=1)
    business_explanation: str = Field(..., min_length=1)

    explanation_json: str = Field(
        ...,
        min_length=2,
        description="JSON de explanation (hard_rule_triggered, thresholds, weights, etc.).",
    )

    candidates_json: str = Field(
        ...,
        min_length=2,
        description="JSON de la lista candidates (top_k_out elementos como máximo).",
    )

    candidate_explanations_json: str = Field(
        ...,
        min_length=2,
        description="JSON de candidate_explanations (desglose por candidato).",
    )

    model_config = {
        "json_schema_extra": {
            "example": {
                "decision": "revisar",
                "score": 0.87,
                "reason": "Score hibrido: comparacion inteligente de nombre, direccion y señales exactas",
                "business_explanation": "Se envio a revision por la combinacion de nombre altamente similar y RFC parcial.",
                "explanation_json": "{\"group_key\":\"P\",\"is_fideicomiso\":false}",
                "candidates_json": "[{\"candidate_customer_number\":\"001\",\"similarity\":0.87}]",
                "candidate_explanations_json": "[{\"candidate_custome` en singular dentro de `model_config`, siguiendo el patrón de [antifraude_rf/schemas.py](models/predictive_classical_ml/antifraude_rf/schemas.py).
le` en singular dentro de `model_config`, siguiendo el patrón de [antifraude_rf/schemas.py](models/predictive_classical_ml/antifraude_rf/schemas.py).

# Adaptaciones internas del framework

## 1. Alinear las firmas

El validador, runner.py, el DAG y los steps deben utilizar las mismas firmas que model.py. Actualmente existen diferencias en load_split y train_model. MLOps debe establecer un contrato único.

## 2. Eliminar preprocessing_contract para este plugin

El motor no construye un ColumnTransformer. El runner y el DAG no deben exigir plugin.preprocessing_contract ni pasarlo a train_model.

## 3. Aceptar DedupePyFunc como resultado

train_step y validate_step están anotados con sklearn.pipeline.Pipeline. Deben aceptar mlflow.pyfunc.PythonModel o Any.

## 4. Registrar la instancia directamente

_as_pyfunc ya devuelve sin envolver cualquier instancia de mlflow.pyfunc.PythonModel. Debe conservarse ese comportamiento:

```python
if isinstance(model, mlflow.pyfunc.PythonModel):
    return model

```

El framework registra la instancia ya cargada. Bento importa el modelo registrado y deserializa esa misma instancia. Bento no vuelve a ejecutar el constructor. Tampoco vuelve a cargar los archivos desde sus rutas. La base, los embeddings, el tokenizer, BERT y las estructuras derivadas viajan como estado serializado dentro del pyfunc. PluginModel no participa en el registro de DedupePyFunc. Por esa razón, en esta propuesta se elimina load_context, no se agrega getstate para excluir recursos pesados y predict no vuelve a llamar a la carga.

## 5. Verificar la serialización completa

El framework registrará una instancia que contiene el DataFrame de clientes, dos matrices de embeddings, Tokenizer y BERT, arreglos derivados y calibración. MLOps debe comprobar que ZenML y cloudpickle pueden materializar y serializar ese objeto sin perder estado, y que el almacenamiento admite su tamaño. No deben pasarse los cinco archivos como artefactos separados en esta variante, porque la clase no utiliza load_context, su contenido ya se encuentra dentro de la instancia.

## 6. Empaquetar dependencias

El entorno necesita numpy, pandas, torch, transformers, rapidfuzz y el lector de parquet. Las versiones utilizadas al serializar deben ser compatibles con el entorno de Bento que deserializa.

## 7. Conservar el código de inferencia

El módulo donde vive DedupePyFunc debe incluir todos los auxiliares utilizados durante predict, o el registro debe incluir explícitamente los módulos adicionales.

## 8. Ajustar esquemas y serving

El conversor actual no admite objetos JSON de tipo object. Mientras no cambie, payload, candidatos y explicaciones complejas viajan como texto JSON. El runtime debe conservar la regla de una salida por entrada y validar el orden de las tuplas contra PredictResponse.

## 9. Probar el modelo serializado

Antes de promover, el gate debe cargar el modelo registrado y ejecutar un ejemplo real. Debe comprobar que el objeto conserva la base, matrices y BERT, que no intenta abrir rutas del entorno de desarrollo, que procesa más de una fila, que devuelve una salida por fila y que la respuesta cumple el esquema.

## 10. Evaluar optimizaciones de tamaño (Alternativa)

Serializar la base, dos matrices y BERT puede producir un modelo de varios gigabytes y duplicar memoria durante registro y carga. Cuándo sí usarías la alternativa: Solo si la prueba demuestra que el pickle completo es inviable, getstate excluye lo pesado, log_model empaqueta los 5 archivos utilizando artifacts, y load_context los abre al cargar en Bento.